# Ensemble Portfolios

Combines every model's predictions into a single ensemble signal, then forms ranked portfolios from it. Each month, each model's prediction is converted to a percentile rank and the ranks are averaged across models; rank-averaging keeps models on a common scale so one model's wider output range cannot dominate the blend. Stocks are then sorted into N_PORT groups and a long-short (top minus bottom) is formed.

Alignment: target_w is already the next-month (t+1) return, built into the panel upstream. Sorting on the ensemble signal at eom and realizing target_w at the same eom is point-in-time correct, so nothing here is lagged.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
# ---- config ----------------------------------------------------------------
from pathlib import Path

# All prediction files to ensemble. Either list them explicitly, or glob a folder.
PRED_FILES = sorted(Path('.').glob('*_predictions.parquet'))   # every *_predictions.parquet here
# ...or set explicitly, e.g.:
'''
PRED_FILES = ['NN1_predictions.parquet','NN2_predictions.parquet',
              'NN3_predictions.parquet','NN4_predictions.parquet',
              'NN5_predictions.parquet',
              'RF_predictions.parquet',
              'RIDGE_predictions.parquet'] # do not include XGB or KNN, they suck
'''
ENSEMBLE   = 'rank'   # 'rank' = avg of per-month percentile ranks (scale-robust,
                      #          recommended when mixing model families);
                      # 'raw'  = avg of raw predictions (only if same scale)
N_PORT     = 10       # 10 = deciles, 5 = quintiles
EQUAL_WT   = True     # True: equal-weight; False + a 'me' column: value-weight
ANNUALIZE  = 12       # monthly data

PRED_FILES = [str(f) for f in PRED_FILES]
assert PRED_FILES, 'no *_predictions.parquet files found -- set PRED_FILES explicitly'
print(f'ensembling {len(PRED_FILES)} models:')
for f in PRED_FILES:
    print('  ', Path(f).stem.replace('_predictions',''))

In [ ]:
# ---- load every model, combine into one ensemble signal --------------------
from functools import reduce

def model_name(path):
    return Path(path).stem.replace('_predictions', '')

frames = []
for f in PRED_FILES:
    d = pd.read_parquet(f)[['permno', 'eom', 'target_w', 'prediction']].copy()
    d['eom'] = pd.to_datetime(d['eom'])
    d = d.dropna(subset=['prediction', 'target_w'])
    if ENSEMBLE == 'rank':
        # per-month percentile rank of this model's prediction -> unitless [0,1]
        d['sig'] = d.groupby('eom')['prediction'].rank(pct=True)
    else:
        d['sig'] = d['prediction']
    frames.append(d[['permno', 'eom', 'target_w', 'sig']]
                  .rename(columns={'sig': model_name(f)}))

# outer-merge on the shared keys so a stock-month present in ANY model survives
MODELS = [model_name(f) for f in PRED_FILES]
preds = reduce(lambda a, b: a.merge(b, on=['permno', 'eom', 'target_w'], how='outer'),
               frames)

# ensemble prediction = mean of the (ranked or raw) per-model signals.
# skipna=True so a stock scored by only some models still gets an average.
preds['prediction'] = preds[MODELS].mean(axis=1, skipna=True)
preds['n_models']   = preds[MODELS].notna().sum(axis=1)
preds = preds.dropna(subset=['prediction', 'target_w'])

print(f"{len(preds):,} stock-months | {preds['eom'].nunique()} months | "
      f"{preds['eom'].min():%Y-%m} to {preds['eom'].max():%Y-%m}")
print(f"coverage: {(preds['n_models']==len(MODELS)).mean():.0%} of rows scored by ALL "
      f"{len(MODELS)} models; mean models/row = {preds['n_models'].mean():.1f}")

### How much do the models agree?

If the models rank stocks identically, ensembling adds nothing; if they disagree, averaging cancels some of each model's idiosyncratic noise. The pairwise correlation of the per-model signals shows the structure — a cluster of highly correlated models (the NNs, typically) effectively counts as one vote against a more independent model such as KNN.


In [ ]:
# pairwise correlation of the per-model (ranked) signals
corr = preds[MODELS].corr()
print('mean pairwise correlation: '
      f'{corr.where(~np.eye(len(MODELS),dtype=bool)).stack().mean():.3f}')
corr.round(2)

In [ ]:
# ---- assign portfolios: percentile rank of prediction WITHIN each month ----
# rank(pct=True) -> (0,1]; ceil(r * N_PORT) buckets into 1..N_PORT.
# method="first" breaks ties by order so bucket sizes stay balanced.
r = preds.groupby("eom")["prediction"].rank(pct=True, method="first")
preds["port"] = np.ceil(r * N_PORT).clip(1, N_PORT).astype(int)

In [ ]:
# ---- monthly return of each portfolio --------------------------------------
# equal-weight = simple mean of target_w within (month, port).
# value-weight: set EQUAL_WT=False and include a 'me' (market-equity) column.
if EQUAL_WT or "me" not in preds.columns:
    port_ret = (preds.groupby(["eom", "port"])["target_w"]
                     .mean().unstack("port"))
else:
    def vw(g):
        return np.average(g["target_w"], weights=g["me"].to_numpy())
    port_ret = preds.groupby(["eom", "port"]).apply(vw).unstack("port")

port_ret.columns = [f"P{c}" for c in port_ret.columns]
port_ret["LS"] = port_ret[f"P{N_PORT}"] - port_ret["P1"]   # long top, short bottom

In [ ]:
# ---- summary stats ---------------------------------------------------------
def stats(x):
    x = x.dropna()
    mu, sd = x.mean(), x.std()
    return pd.Series({
        "ann_return":  mu * ANNUALIZE,
        "ann_vol":     sd * np.sqrt(ANNUALIZE),
        "sharpe":      (mu / sd * np.sqrt(ANNUALIZE)) if sd > 0 else np.nan,
        "hit_rate":    (x > 0).mean(),
        "worst_month": x.min(),
    })

summary = port_ret.apply(stats).T
summary.round(4)

In [ ]:
# ---- cumulative growth of $1 (compounded) ----------------------------------
cum = (1 + port_ret).cumprod()

In [ ]:
# ---- plots -----------------------------------------------------------------
# Two cumulative views each for LS and the deciles: linear ($ value as you'd
# actually read it) and log (so the early years aren't visually crushed by
# late compounding). 3 rows x 2 cols.
fig, axes = plt.subplots(3, 2, figsize=(15, 15))
cmap = plt.cm.RdYlGn(np.linspace(0, 1, N_PORT))

# row 1: long-short growth of $1 -- linear (left) and log (right)
for ax, logy in ((axes[0, 0], False), (axes[0, 1], True)):
    ax.plot(cum.index, cum["LS"], lw=1.8, color="black")
    if logy:
        ax.set_yscale("log")
    ax.set_title(f"Long-Short (P{N_PORT} - P1): growth of $1"
                 + (" (log)" if logy else " (linear $)"))
    ax.set_ylabel("cumulative value"); ax.grid(True, alpha=0.3)

# row 2: each portfolio's growth of $1 -- linear (left) and log (right)
for ax, logy in ((axes[1, 0], False), (axes[1, 1], True)):
    for i in range(1, N_PORT + 1):
        ax.plot(cum.index, cum[f"P{i}"], lw=1.0, color=cmap[i-1], label=f"P{i}")
    if logy:
        ax.set_yscale("log")
    ax.set_title("Each portfolio: growth of $1"
                 + (" (log)" if logy else " (linear $)"))
    ax.set_ylabel("cumulative value"); ax.grid(True, alpha=0.3)
    ax.legend(ncol=2, fontsize=8)

# row 3 left: mean annualized return by portfolio (monotonicity test)
ax = axes[2, 0]
means = port_ret[[f"P{i}" for i in range(1, N_PORT+1)]].mean() * ANNUALIZE
colors = ["firebrick" if v < 0 else "steelblue" for v in means]
ax.bar(range(1, N_PORT+1), means.values, color=colors)
ax.axhline(0, color="grey", lw=0.8)
ax.set_title("Annualized mean return by portfolio\n(monotone rise = signal)")
ax.set_xlabel(f"portfolio (1 = lowest pred, {N_PORT} = highest)")
ax.set_ylabel("annualized return")
ax.set_xticks(range(1, N_PORT+1)); ax.grid(True, alpha=0.3, axis="y")

# row 3 right: long-short drawdown
ax = axes[2, 1]
dd = cum["LS"] / cum["LS"].cummax() - 1
ax.fill_between(dd.index, dd.values, 0, color="firebrick", alpha=0.4)
ax.set_title("Long-Short drawdown")
ax.set_ylabel("drawdown"); ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# ---- save the monthly portfolio return series ------------------------------
port_ret.to_parquet("portfolio_returns.parquet")
print("saved -> portfolio_returns.parquet")

**Reading the long-short.** The ensemble signal produces an annualized long-short return of ____ with a Sharpe of ____ (t-stat ____ over ____ months). Return climbs [monotonically / unevenly] from P1 to P10, which [supports / undercuts] the read that the signal orders stocks correctly rather than acting on a single extreme group. The maximum drawdown of ____ and hit rate of ____% describe how the spread was earned. Because this is the ensemble, compare its Sharpe against the best single model's: an ensemble that only matches its best member is buying robustness, not extra return.


### Is the spread real?

A rising cumulative line is not evidence of skill on its own. The monthly long-short series below supports a t-test on its mean, and — given factor data — a regression on market, size, value, and momentum to check for alpha beyond the known premia.


In [ ]:
# quick significance check on the long-short mean
from scipy import stats as sps
ls = port_ret["LS"].dropna()
t, p = sps.ttest_1samp(ls, 0)
print(f"LS monthly mean {ls.mean():.4%} | t-stat {t:.2f} | p-value {p:.4g} "
      f"| n={len(ls)} months")

## Sub-period comparison: first half vs second half

The realized long-short returns are split into two halves of equal length by month count (the median month divides them), so each half is tested on the same number of observations. This checks whether the strategy's return and risk held up across the two eras; it does not retrain the model. The exact dates of each half print with the split below.

A signal that is real but time-varying often looks strong in one half and weak in the other. A signal that holds up in both is the more convincing.


In [ ]:
# ---- split realized returns into two EQUAL halves by MONTH COUNT -----------
# equal month split: first N/2 months vs last N/2 months (differ by at most 1
# when the total is odd), regardless of calendar year. This makes H1 and H2
# equal-length samples rather than splitting on a year boundary.
port_ret = port_ret.sort_index()
n_months = len(port_ret)
mid = n_months // 2
h1 = port_ret.iloc[:mid]
h2 = port_ret.iloc[mid:]
print(f"{n_months} months -> H1 {len(h1)} months, H2 {len(h2)} months")
print(f"H1: {h1.index.min():%Y-%m} to {h1.index.max():%Y-%m}")
print(f"H2: {h2.index.min():%Y-%m} to {h2.index.max():%Y-%m}")

In [ ]:
# ---- side-by-side return/risk stats for the long-short ---------------------
def period_stats(x):
    x = x.dropna()
    mu, sd = x.mean(), x.std()
    dd = (1 + x).cumprod()
    max_dd = (dd / dd.cummax() - 1).min()
    from scipy import stats as sps
    t, p = sps.ttest_1samp(x, 0)
    return pd.Series({
        "months":      len(x),
        "ann_return":  mu * ANNUALIZE,
        "ann_vol":     sd * np.sqrt(ANNUALIZE),
        "sharpe":      mu / sd * np.sqrt(ANNUALIZE) if sd > 0 else np.nan,
        "hit_rate":    (x > 0).mean(),
        "max_drawdown": max_dd,
        "t_stat":      t,
        "p_value":     p,
    })

compare = pd.DataFrame({
    "H1 (" + str(h1.index.year.min()) + "-" + str(h1.index.year.max()) + ")": period_stats(h1["LS"]),
    "H2 (" + str(h2.index.year.min()) + "-" + str(h2.index.year.max()) + ")": period_stats(h2["LS"]),
    "Full": period_stats(port_ret["LS"]),
})
compare.round(4)

In [ ]:
# ---- compare the two halves visually ---------------------------------------
# Each half's growth of $1 is REBASED to start at 1 at the start of that half,
# so the two eras are compared on equal footing (H2 doesn't inherit H1's level).
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# 1. LS growth of $1, each half rebased to 1, linear
ax = axes[0, 0]
for h, lab, col in ((h1, f"H1 {h1.index.year.min()}-{h1.index.year.max()}", "steelblue"),
                    (h2, f"H2 {h2.index.year.min()}-{h2.index.year.max()}", "firebrick")):
    g = (1 + h["LS"]).cumprod()
    ax.plot(np.arange(len(g)), g.values, lw=1.6, color=col, label=lab)
ax.set_title("Long-Short growth of $1, each half rebased (linear)")
ax.set_xlabel("months into the half"); ax.set_ylabel("cumulative value")
ax.legend(); ax.grid(True, alpha=0.3)

# 2. same, log axis
ax = axes[0, 1]
for h, lab, col in ((h1, "H1", "steelblue"), (h2, "H2", "firebrick")):
    g = (1 + h["LS"]).cumprod()
    ax.plot(np.arange(len(g)), g.values, lw=1.6, color=col, label=lab)
ax.set_yscale("log")
ax.set_title("Long-Short growth of $1, each half rebased (log)")
ax.set_xlabel("months into the half"); ax.set_ylabel("cumulative value")
ax.legend(); ax.grid(True, alpha=0.3)

# 3. decile monotonicity in each half (does the P1->P10 rise survive both?)
ax = axes[1, 0]
cols = [f"P{i}" for i in range(1, N_PORT+1)]
w = 0.4
ax.bar(np.arange(N_PORT) - w/2, h1[cols].mean()*ANNUALIZE, w,
       color="steelblue", label="H1")
ax.bar(np.arange(N_PORT) + w/2, h2[cols].mean()*ANNUALIZE, w,
       color="firebrick", label="H2")
ax.axhline(0, color="grey", lw=0.8)
ax.set_title("Annualized return by portfolio, per half")
ax.set_xlabel("portfolio"); ax.set_ylabel("annualized return")
ax.set_xticks(range(N_PORT)); ax.set_xticklabels(range(1, N_PORT+1))
ax.legend(); ax.grid(True, alpha=0.3, axis="y")

# 4. annualized return + vol bars, H1 vs H2
ax = axes[1, 1]
metrics = pd.DataFrame({
    "H1": [h1["LS"].mean()*ANNUALIZE, h1["LS"].std()*np.sqrt(ANNUALIZE)],
    "H2": [h2["LS"].mean()*ANNUALIZE, h2["LS"].std()*np.sqrt(ANNUALIZE)],
}, index=["ann_return", "ann_vol"])
metrics.plot(kind="bar", ax=ax, color=["steelblue", "firebrick"])
ax.set_title("Long-Short return & vol: H1 vs H2")
ax.set_ylabel("annualized"); ax.set_xticklabels(metrics.index, rotation=0)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

**Reading the two halves.** The long-short earned ____ annualized in the first half against ____ in the second, with Sharpes of ____ and ____. The profile is [broadly stable / clearly weaker in H2 / clearly weaker in H1], which suggests the edge [persists / has decayed / strengthened] over time. The per-portfolio bars show the P1-to-P10 rise [surviving / breaking down] in the weaker half. A large gap between halves is the main caution to flag: it points to a time-varying signal whose full-sample number oversells its recent performance.


### Long-only (top decile), halves side by side

For a no-shorting account the tradable version is the top group (P10) held alone. Each half is drawn on its own axis rather than overlaid. P1 is shown alongside to locate where the long-short spread comes from: if most of it sits in P1 (the short leg), the strategy leans on shorting and the long-only P10 will be noticeably weaker.


In [ ]:
# ---- long-only P10 (and P1) growth of $1, halves in separate panels ---------
top, bot = f"P{N_PORT}", "P1"

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5), sharey=True)
halves = [("H1", h1, f"{h1.index.year.min()}-{h1.index.year.max()}"),
          ("H2", h2, f"{h2.index.year.min()}-{h2.index.year.max()}")]

for ax, (tag, h, span) in zip(axes, halves):
    g_top = (1 + h[top]).cumprod()
    g_bot = (1 + h[bot]).cumprod()
    x = np.arange(len(h)) # months into the half
    ax.plot(x, g_top.values, lw=1.8, color="seagreen",
            label=f"{top} (long top)")
    ax.plot(x, g_bot.values, lw=1.4, color="firebrick", alpha=0.8,
            label=f"{bot} (long bottom)")
    ax.axhline(1, color="grey", lw=0.8, ls=":")
    ax.set_title(f"{tag}: {span}  — long-only growth of $1")
    ax.set_xlabel("months into the half"); ax.grid(True, alpha=0.3)
    ax.legend()
axes[0].set_ylabel("cumulative value (rebased to 1)")
plt.tight_layout(); plt.show()

# ---- long-only stats table, both legs x both halves ------------------------
lo = pd.DataFrame({
    f"{top} H1": period_stats(h1[top]), f"{top} H2": period_stats(h2[top]),
    f"{bot} H1": period_stats(h1[bot]), f"{bot} H2": period_stats(h2[bot]),
})
lo.round(4)

**Reading the long-only legs.** Held alone, P10 returned ____ annualized (Sharpe ____), versus the full long-short's ____. P10 retains [most / roughly half / little] of the spread, which means the strategy's edge is [mostly on the long side and workable without shorting / substantially in the short leg and hard to capture in a no-shorting account]. For a Roth IRA, where shorting is not permitted, this long-only figure is the one that actually applies.
